# 6.1. Layers and Modules
D2L의 Layers and Modules장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Layer, Module, Model의 관계

지금까지 우리는 `nn.Linear`같은 layer를 사용해서 신경망을 만들었다. 하지만 실제 신경망은 여러 layer를 묶어서 구성한다.

```text
입력 X
 ↓
Linear
 ↓
ReLU
 ↓
Linear
 ↓
출력
```

PyTorch에서는 이런 구성 요소들을 모두 Module이라는 단위로 다룬다

Module은 하나의 layer일수도 있고, 여러 layer를 묶은 블록일 수도 있고, 전체 모델일 수도 있다.

큰 신경망은 작은 Module들을 조립해서 만든다고 생각하면 된다.

D2L에서 강조하는 건   
layer -> 여러 layer를 묶은 module -> 여러 module을 묶은 더 큰 model이라는 계층 구조이다.

복잡한 ResNet 같은 모델도 결국 이 구조를 반복해서 쌓는다. 

## 2. 가장 간단한 모델구성 nn.Sequential

`nn.Sequential`은 layer들을 순서대로 실행하는 모델을 쉽게 만드는 방법이다.

예를 들어서 아래 MLP를 생각해보면

```text
[batch_size, 20]

 ↓ Linear(20, 256)

[batch_size, 256]

 ↓ ReLU

[batch_size, 256]

 ↓ Linear(256, 10)

[batch_size, 10]
```

nn.Sequential 안에 layer들을 실행할 순서대로 넣으면 된다.

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

X = torch.rand(2, 20)

net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

y_hat = net(X)

print("입력 shape:", X.shape)
print("출력 shape:", y_hat.shape)

입력 shape: torch.Size([2, 20])
출력 shape: torch.Size([2, 10])


Sequential은 내부적으로 아래 작업을 한다.

```text
X
 ↓
Linear(20, 256)
 ↓
ReLU
 ↓
Linear(256, 10)
 ↓
출력
```
이전 layer의 출력이 다음 layer의 입력으로 들어간다.

$$
X_1 = Linear_1(X)
$$

$$
X_2 = ReLU(X_1)
$$

$$
Y = Linear_2(X_2)
$$

`nn.Sequential`은 이 과정을 자동으로 연결해준다. `Sequential` 자체도 `nn.Module`이고, 내부에 다른 `Module`들을 순서대로 보관하고 실행한다고 한다.

## 3. MLP Module 만들기

`nn.Sequential`을 사용하지 않고 우리가 직접 모델을 정의할 수도 있다.

PyTorch에서 직접 모델을 만들 때는 보통 다음 형태를 사용한다고 한다.

```py
class 모델이름(nn.Module):

    def __init__(self): # 모델이 어떤 layer를 가지고 있을지
        super().__init__()

        # 사용할 layer 정의

    def forward(self, X): # 입력 데이터가 그 layer들을 어떤 순서와 방식으로 통과할지 정의

        # 데이터가 어떻게 layer들을 통과할지 정의
```

In [3]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        X = self.hidden(X)
        X = F.relu(X)
        X = self.out(X)

        return X

In [4]:
net = MLP()

y_hat = net(X)

print(y_hat.shape)

torch.Size([2, 10])


실행되는 과정은 이렇다

```text
X [2, 20]

 ↓ self.hidden(X)

[2, 256]

 ↓ ReLU

[2, 256]

 ↓ self.out(X)

[2, 10]
```

전에 썼던 `nn.Sequential`과 결과적으로 같은 MLP이다. 

D2L에서도 사용자 정의 Module에서 기본적으로 `__init__`에서 layer를 정의하고 `forward`에서 계산 흐름을 정의하면, 역전파와 gradient 계산 등은 PyTorch의 autograd가 처리한다고 말한다.

## 4. net(X)와 forward()

우리는 모델에 데이터를 넣을 때 이렇게 쓴다

    y_hat = net(X)

우리가 만든건 forward()인데 net.forward(X)로 쓰지 않을까?

PyTorch의 `nn.Module`에는 `__call__()`이라는 기능이 있다.

net(X)를 실행하면 내부적으로 

    net(X) -> nn.Module.__call__() -> forward(X)

흐름으로 forward()가 호출된다.

이 구조 덕분에 PyTorch가 forward hook 등 Module에 필요한 추가 동작들도 함계 처리할 수 있으므로 일반적인 모델 호출은 `net(X)`를 사용하는 것이 맞다고 한다. 

D2L에서도 `net(X)`가 Module의 호출 매커니즘을 통해 forward 계산을 수행한다고 설명한다.

## 5. Sequential은 내부적으로 어떻게 동작할까?

`Sequential`의 핵심 아이디어는 이렇다.

```py
for layer in layers:
    X = layer(X)
```

여러 Module을 저장해 놓고 처음부터 끝까지 순서대로 실행한다. 간단한 Sequential을 만들어보면

In [5]:
class MySequential(nn.Module):

    def __init__(self, *layers):
        super().__init__()

        self.layers = nn.ModuleList(layers)

    def forward(self, X):

        for layer in self.layers:
            X = layer(X)

        return X

In [6]:
net = MySequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

y_hat = net(X)

print(y_hat.shape)

torch.Size([2, 10])


D2L에서는 `add_module()`과 `children()`을 사용해 직접 `Sequential`구조를 구현하는데 우리는 직관적인 `nn.ModuleList`를 썼다. 

일반 python `list`와 다르게 `ModuleList`에 들어간 layer들은 PyTorch에 하위 module로 등록되어 `parameters()`, `to(device)` 등의 관리 대상이 된다.

## 6. 왜 굳이 Module을 만들까?

단순한 모델이면 `nn.Sequential`이면 충분하다.

예를 들어서 이런 동작이 있을 수도 있다.
```py
if 조건:
    layer(X)
else:
    layer2(X)


while 조건:
    X = X / 2
```

혹은 하나의layer를 여러 번 쓸 수도있다.

이런경우에 `nn.Sequential`보다는 직접 만들어 `forward()`에서 계산 과정을 작성하는 것이 훨씬 자유롭다고 한다.

D2L에서 보여주려는 내용은 이것이다. `forward()`에는 단순한 layer 연결뿐 아니라 일반적인 Python 제어 흐름이나 수학 연산을 넣을 수 있다.

## 7. forward 안에 일반 코드 넣기

D2L에 `FixedHiddenMLP`가 이것을 보여준다.

In [ ]:
class FixedHiddenMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.linear = nn.Linear(20, 20)

        # 학습되지 않는 고정 행렬
        self.register_buffer(
            "rand_weight",
            torch.rand(20, 20)
        )

    def forward(self, X):

        # 학습 가능한 Linear
        X = self.linear(X)

        # 일반 행렬 연산
        X = X @ self.rand_weight + 1
        X = F.relu(X)

        # 같은 Linear를 다시 사용 (같은 w, b 공유)
        X = self.linear(X)

        # 일반 Python 제어문
        while X.abs().sum() > 1:
            X = X / 2

        return X.sum()

In [ ]:
net = FixedHiddenMLP()

output = net(X)

print(output)
print(output.shape) # 출력이 하나의 숫자이기 때문에

tensor(0.1076, grad_fn=<SumBackward0>)
torch.Size([])


D2L이 말하고자 하는건 forward는 layer를 단순히 나열하는게 아니라 모델의 전체 계산 과정을 정의하는 함수라는 걸 말하는 것 같다. 원문 `FixedHiddenMLP`에도 학습되지 않는 고정행렬, layer 재사용, while 문등을 한 forward에 넣어 Module이 얼마나 유연한지 보여준다.

D2L에서는 단순 tensor인 rand_weight를 사용하지만, 실전 PyTorch에서는 학습 대상은 아니면서 모델과 함께 CPU/GPU 이동 및 저장이 필요한 tensor라면 register_buffer()를 사용하는 편이 적절하다고 한다. 등록된 parameters와 buffers는 Module의 장치 변환 등의 관리 대상이 된다.

## 8. Module 안에 Module 넣기

Module의 중요한 점은 중첩할 수 있는 것이다.

예를 들어서 작은 신경망 블록을 만들고 그 블록을 더 큰 신경망 안에서 하나의 layer처럼 사용할 수 있다.

```text
모델
 ├─ 작은 Module
 │    ├─ Linear
 │    ├─ ReLU
 │    ├─ Linear
 │    └─ ReLU
 ├─ Linear
 │
 └─ 다른 Module
```
실제 딥러닝 모델들이 이런 식으로 작은 블록을 반복해서 쌓아 만들어진다.

In [9]:
class HiddenBlock(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        self.out = nn.Linear(32, 16)

    def forward(self, X):
        X = self.net(X)
        X = self.out(X)

        return X

In [15]:
block = HiddenBlock()

output = block(X)

print(output.shape)

torch.Size([2, 16])


In [ ]:
big_model = nn.Sequential(
    HiddenBlock(), # 입력 [2, 20] -> HiddenBlack() -> [2, 16]
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)

output = big_model(X)

print(output.shape)

torch.Size([2, 10])


`HiddenBlock`도 바깥 모델에서는 하나의 Module처럼 사용할 수 있다. 

D2L의 마지막 예제도 `NestMLP`라는 Module을 만든 뒤 다른 Module들과 다시 조립한다.

## 9. 오늘의 정리

- PyTorch에서는 layer, 여러 layer로 이루어진 블록, 전체 모델을 모두 `nn.Module`로 표현할 수 있다.
- `nn.Linear`, `nn.ReLU`, `nn.Sequential`도 모두 Module이다.
- 직접 모델을 만들려면 `nn.Module`을 상속한다.
- `__init__()`에서는 모델이 사용할 layer들을 정의한다.
- `forward()`에서는 입력 데이터가 어떤 순서와 방식으로 계산될지 정의한다.
- `net(X)`를 실행하면 PyTorch가 내부적으로 `forward(X)`를 사용하여 순전파를 수행한다.
- `nn.Sequential`은 여러 Module을 순서대로 실행해주는 편리한 Module이다.
- `forward()` 안에는 layer뿐 아니라 tensor 연산, 조건문, 반복문 등 일반적인 코드도 넣을 수 있다.
- 하나의 Module 안에 다른 Module을 넣을 수 있으며, 이런 작은 블록들을 조립하여 복잡한 신경망을 만든다.